In [2]:
import os
import pickle
import math
import cv2
import numpy as np
import imutils
from imutils import paths
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import torch
from torch import nn, optim
from torch.nn import functional as f
from torch.utils.data import DataLoader, TensorDataset

# ==========================================
# Part 1: Helper Functions (from lab_3_helpers.py)
# ==========================================

def resize_to_fit(image, width, height):
    """
    A helper function to resize an image to fit within a given size.
    """
    (h, w) = image.shape[:2]

    if w > h:
        image = imutils.resize(image, width=width)
    else:
        image = imutils.resize(image, height=height)

    padW = int((width - image.shape[1]) / 2.0)
    padH = int((height - image.shape[0]) / 2.0)

    image = cv2.copyMakeBorder(image, padH, padH, padW, padW,
        cv2.BORDER_REPLICATE)
    image = cv2.resize(image, (width, height))
    return image

def get_torch_device():
    if torch.cuda.is_available():
        device = "cuda"
    elif torch.backends.mps.is_available():
        device = "mps"
    else:
        device = "cpu"
    return torch.device(device)

def unzip(iterable):
    """ Unzip iterable of tuples into tuple of lists. """
    unzip_lists = None
    n_lists = None
    
    for values in iterable:
        n_values = len(values)
        if unzip_lists is None:
            unzip_lists = tuple(([value] for value in values))
            n_lists = n_values
        else:
            if n_values!=n_lists:
                raise ValueError(f"Expect tuple of {n_lists} values, got {n_values}")
            for value, unzip_list in zip(values, unzip_lists):
                unzip_list.append(value)
    return unzip_lists

def group_every(iterable, n_elem=1):
    """ Group every N elements into a list and yield that list. """
    group = []
    for value in iterable:
        group.append(value)
        if len(group)>=n_elem:
            yield group
            group = []
    if group:
        yield group

# ==========================================
# Part 2: Preprocessing Logic
# ==========================================

def extract_captcha_text(image_path):
    image_file_name = os.path.basename(image_path)
    return os.path.splitext(image_file_name)[0]

def load_transform_image(image_path):
    image = cv2.imread(image_path)
    image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    image_padded = cv2.copyMakeBorder(image_gray, 8, 8, 8, 8, cv2.BORDER_REPLICATE)
    return image_padded

def extract_chars(image):
    """ Find contours and extract characters inside each CAPTCHA. """
    image_bw = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)[1]
    contours = cv2.findContours(image_bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]

    char_regions = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        # If bounding box is too wide, split it
        if w / h > 1.25:
            half_width = int(w / 2)
            char_regions.append((x, y, half_width, h))
            char_regions.append((x + half_width, y, half_width, h))
        else:
            char_regions.append((x, y, w, h))

    # All CAPTCHAs should have 4 characters
    if len(char_regions) != 4:
        return None

    # Sort by X coordinate
    char_regions.sort(key=lambda x: x[0])

    char_images = []
    for x, y, w, h in char_regions:
        char_image = image[y - 2:y + h + 2, x - 2:x + w + 2]
        char_images.append(char_image)      
    return char_images

def save_chars(char_images, captcha_text, save_dir, char_counts):
    for char_image, char in zip(char_images, captcha_text):
        save_path = os.path.join(save_dir, char)
        os.makedirs(save_path, exist_ok=True)
        char_count = char_counts.get(char, 1)
        char_image_path = os.path.join(save_path, f"{char_count}.png")
        cv2.imwrite(char_image_path, char_image)
        char_counts[char] = char_count + 1

def make_feature(image):
    # Resize to 20x20
    image_resized = resize_to_fit(image, 20, 20)
    # Add channel dimension (1, 20, 20)
    feature = image_resized[None, ...]
    return feature

def make_feature_label(image_path):
    feature = make_feature(cv2.imread(image_path, cv2.COLOR_BGR2GRAY))
    label = image_path.split(os.path.sep)[-2]
    return feature, label

# ==========================================
# Part 3: Main Execution Flow
# ==========================================

if __name__ == "__main__":
    # --- Configuration ---
    CAPTCHA_IMAGE_FOLDER = "./captcha-images"
    TVT_SPLIT_SEED = 31528476
    CHAR_IMAGE_FOLDER = f"./char-images-{TVT_SPLIT_SEED}"
    LABELS_PATH = "./labels.pkl"
    MODEL_WEIGHTS_PATH = "./captcha-model-weights.pt"
    BATCH_SIZE = 32
    N_EPOCHS = 20
    FORCE_TRAINING = True # Set to True to retrain

    # Ensure dataset exists
    if not os.path.exists(CAPTCHA_IMAGE_FOLDER):
        print(f"Error: Dataset folder '{CAPTCHA_IMAGE_FOLDER}' not found.")
        print("Please unzip the captcha-images dataset into the current directory.")
        exit(1)

    print(">>> 1. Loading and Splitting CAPTCHA Images...")
    captcha_image_paths = list(paths.list_images(CAPTCHA_IMAGE_FOLDER))
    captcha_texts = [extract_captcha_text(p) for p in captcha_image_paths]
    captcha_images = [load_transform_image(p) for p in captcha_image_paths]

    # Split into Train/Val and Test (Test is kept aside for final end-to-end eval)
    captcha_images_tv, captcha_images_test, captcha_texts_tv, captcha_texts_test = train_test_split(
        captcha_images, captcha_texts, test_size=0.2, random_state=TVT_SPLIT_SEED
    )
    print(f"Train-Val Set: {len(captcha_texts_tv)}, Test Set: {len(captcha_texts_test)}")

    print(">>> 2. Extracting Single Characters...")
    char_counts = {}
    if not os.path.exists(CHAR_IMAGE_FOLDER):
        for captcha_image, captcha_text in zip(captcha_images_tv, captcha_texts_tv):
            char_images = extract_chars(captcha_image)
            if char_images is None:
                continue
            save_chars(char_images, captcha_text, CHAR_IMAGE_FOLDER, char_counts)
    else:
        print(f"Folder {CHAR_IMAGE_FOLDER} already exists, skipping extraction.")

    print(">>> 3. Creating Features and Labels...")
    features_tv, labels_tv = unzip((
        make_feature_label(image_path) for image_path in paths.list_images(CHAR_IMAGE_FOLDER)
    ))

    # Normalize to [0, 1]
    features_tv = np.array(features_tv, dtype=np.float32) / 255.0

    # Encode labels
    le = LabelEncoder()
    label_indices_tv = le.fit_transform(labels_tv)
    n_classes = len(le.classes_)
    print(f"Number of classes: {n_classes}")

    # Split Train/Val for the Neural Network
    X_train, X_vali, y_train, y_vali = train_test_split(
        features_tv, label_indices_tv, test_size=0.25, random_state=955996
    )

# Save Label Encoder
    with open(LABELS_PATH, "wb") as file:  
        pickle.dump(le, file)              

    # ==========================================
    # Part 4: Model Definition (TODO 1)
    # ==========================================
    print(">>> 4. Building PyTorch Model...")
    torch_device = get_torch_device()
    print(f"Using device: {torch_device}")

    # [ TODO Completed ] Define CNN Model
    model = nn.Sequential(
        # Block 1
        # Input: (Batch, 1, 20, 20)
        # Conv2d: 1 -> 20, kernel 5x5, padding 2 (keeps size 20x20)
        nn.Conv2d(1, 20, kernel_size=5, padding=2),
        nn.ReLU(),
        # MaxPool: 2x2 (size becomes 10x10)
        nn.MaxPool2d(2, 2),
        
        # Block 2
        # Input: (Batch, 20, 10, 10)
        # Conv2d: 20 -> 50, kernel 5x5, padding 2 (keeps size 10x10)
        nn.Conv2d(20, 50, kernel_size=5, padding=2),
        nn.ReLU(),
        # MaxPool: 2x2 (size becomes 5x5)
        nn.MaxPool2d(2, 2),
        
        # Flatten
        # Input: (Batch, 50, 5, 5) -> Output: (Batch, 1250)
        nn.Flatten(),
        
        # Fully Connected 1
        nn.Linear(50 * 5 * 5, 500),
        nn.ReLU(),
        
        # Output Layer
        # Output: (Batch, n_classes)
        nn.Linear(500, n_classes)
    )

    model = model.to(torch_device)
    optimizer = optim.AdamW(model.parameters(), lr=0.0001)

    # Data Loaders
    dataset_train = TensorDataset(torch.as_tensor(X_train), torch.as_tensor(y_train))
    dataset_vali = TensorDataset(torch.as_tensor(X_vali), torch.as_tensor(y_vali))
    loader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
    loader_vali = DataLoader(dataset_vali, batch_size=BATCH_SIZE)

    # ==========================================
    # Part 5: Training Loop (TODO 2)
    # ==========================================
    if FORCE_TRAINING or not os.path.exists(MODEL_WEIGHTS_PATH):
        print(">>> 5. Starting Training...")
        for i in range(N_EPOCHS):
            loss_train = 0.
            model.train() # Set to training mode

            # [ TODO Completed ] Training Loop
            for X_batch, y_batch in tqdm(loader_train, desc=f"Epoch {i+1}/{N_EPOCHS} [Train]"):
                X_batch = X_batch.to(torch_device)
                y_batch = y_batch.to(torch_device)

                # 1. Zero gradients
                optimizer.zero_grad()
                
                # 2. Forward pass
                outputs = model(X_batch)
                
                # 3. Compute loss (CrossEntropy includes Softmax)
                loss = f.cross_entropy(outputs, y_batch)
                
                # 4. Backward pass
                loss.backward()
                
                # 5. Optimization step
                optimizer.step()
                
                # 6. Accumulate loss (detach to prevent graph buildup, move to cpu for numpy math)
                loss_train += loss.detach().cpu().item()

            loss_train /= len(loader_train)
            
            # Validation
            loss_vali = 0.
            model.eval() # Set to evaluation mode

            with torch.no_grad(): # Disable gradient calculation for validation
                for X_batch, y_batch in tqdm(loader_vali, desc=f"Epoch {i+1}/{N_EPOCHS} [Val]"):
                    X_batch = X_batch.to(torch_device)
                    y_batch = y_batch.to(torch_device)
                    
                    outputs = model(X_batch)
                    loss = f.cross_entropy(outputs, y_batch)
                    loss_vali += loss.detach().cpu().item()

            loss_vali /= len(loader_vali)
            print(f"Epoch {i+1}: Train Loss: {loss_train:.4f}, Val Loss: {loss_vali:.4f}")

        print(f"Saving model to {MODEL_WEIGHTS_PATH}...")
        torch.save(model.state_dict(), MODEL_WEIGHTS_PATH)
    else:
        print("Loading pre-trained model...")
        model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=torch_device))

    # ==========================================
    # Part 6: Evaluation (TODO 3)
    # ==========================================
    print(">>> 6. Evaluating on Test Set...")
    
    # Prepare dummy for failures
    DUMMY_CHAR_IMAGES = np.zeros((4, 20, 20, 1))
    extract_failed_indices = []
    char_images_test = []

    for i, captcha_image in enumerate(captcha_images_test):
        chars = extract_chars(captcha_image)
        if chars:
            char_images_test.extend(chars)
        else:
            extract_failed_indices.append(i)
            char_images_test.extend(DUMMY_CHAR_IMAGES)

    features_test = [make_feature(c) for c in char_images_test]
    features_test = np.array(features_test, dtype=np.float32) / 255.0

    # [ TODO Completed ] Evaluation logic
    model.eval()
    with torch.no_grad():
        # 1. Convert to tensor and move to device
        X_test_tensor = torch.as_tensor(features_test).to(torch_device)
        
        # 2. Predict
        logits = model(X_test_tensor)
        
        # Get index of highest probability
        predictions_indices = torch.argmax(logits, dim=1)
        
        # Move back to CPU for sklearn LabelEncoder
        predictions_indices = predictions_indices.cpu().numpy()
        
        # 3. Decode labels
        preds_test = le.inverse_transform(predictions_indices)

    # Reconstruct predictions into CAPTCHA strings
    preds_test_strings = ["".join(chars) for chars in group_every(preds_test, 4)]
    
    # Mark failed extractions
    for i in extract_failed_indices:
        preds_test_strings[i] = "-"

    # Calculate Accuracy
    n_correct = 0
    for pred, actual in zip(preds_test_strings, captcha_texts_test):
        if pred == actual:
            n_correct += 1
            
    print(f"Test Accuracy: {n_correct}/{len(captcha_texts_test)} ({n_correct/len(captcha_texts_test):.2%})")
    print("Done.")

>>> 1. Loading and Splitting CAPTCHA Images...
Train-Val Set: 908, Test Set: 228
>>> 2. Extracting Single Characters...
Folder ./char-images-31528476 already exists, skipping extraction.
>>> 3. Creating Features and Labels...
Number of classes: 32
>>> 4. Building PyTorch Model...
Using device: cpu
>>> 5. Starting Training...


Epoch 1/20 [Val]: 100%|██████████| 28/28 [00:00<00:00, 193.22it/s]


Epoch 1: Train Loss: 3.3795, Val Loss: 3.2931


Epoch 2/20 [Val]: 100%|██████████| 28/28 [00:00<00:00, 195.16it/s]


Epoch 2: Train Loss: 3.0361, Val Loss: 2.6463


...

Epoch 18: Train Loss: 0.0515, Val Loss: 0.1654


Epoch 19/20 [Val]: 100%|██████████| 28/28 [00:00<00:00, 246.58it/s]


Epoch 19: Train Loss: 0.0443, Val Loss: 0.1656


Epoch 20/20 [Val]: 100%|██████████| 28/28 [00:00<00:00, 227.20it/s]


Epoch 20: Train Loss: 0.0379, Val Loss: 0.1558
Saving model to ./captcha-model-weights.pt...
>>> 6. Evaluating on Test Set...
Test Accuracy: 208/228 (91.23%)
Done.
